In [0]:
from pyspark.sql import functions as F

##Load tables

In [0]:
bronze = spark.table("workspace.bronze.transactions")
silver = spark.table("workspace.silver.transactions")
quarantine = spark.table("workspace.silver.transaction_quarantine")
balances = spark.table("workspace.gold.account_balances")
fraud = spark.table("workspace.gold.fraud_alerts")
daily = spark.table("workspace.gold.daily_transaction_metrics")

##Pipeline record counts

In [0]:
quality_summary = [
    ("Bronze Transactions", bronze.count()),
    ("Silver Transactions", silver.count()),
    ("Quarantine Transactions", quarantine.count()),
    ("Account Balances", balances.count()),
    ("Fraud Alerts", fraud.count()),
    ("Daily KPI Records", daily.count())
]

quality_df = spark.createDataFrame(
    quality_summary,
    ["dataset", "record_count"]
)

display(quality_df)

dataset,record_count
Bronze Transactions,5010000
Silver Transactions,4681541
Quarantine Transactions,318459
Account Balances,135058
Fraud Alerts,1619023
Daily KPI Records,395


##Silver duplicate check

In [0]:
silver_duplicate_ids = (
    silver
    .groupBy("txn_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Silver duplicate transaction IDs:",
    silver_duplicate_ids
)

Silver duplicate transaction IDs: 0


##Silver validation check

In [0]:
invalid_amounts = silver.filter(
    F.col("amount") <= 0
).count()

invalid_currency = silver.filter(
    ~F.col("currency").isin("INR", "USD", "EUR", "GBP")
).count()

invalid_txn_type = silver.filter(
    ~F.col("txn_type").isin("DEBIT", "CREDIT")
).count()

non_success = silver.filter(
    F.col("status") != "SUCCESS"
).count()

print("Invalid amounts:", invalid_amounts)
print("Invalid currencies:", invalid_currency)
print("Invalid transaction types:", invalid_txn_type)
print("Non-success transactions:", non_success)

Invalid amounts: 0
Invalid currencies: 0
Invalid transaction types: 0
Non-success transactions: 0


##Create a formal DQ report

In [0]:
dq_results = [
    ("Silver duplicate txn_id", silver_duplicate_ids),
    ("Invalid amounts in Silver", invalid_amounts),
    ("Invalid currencies in Silver", invalid_currency),
    ("Invalid txn types in Silver", invalid_txn_type),
    ("Non-success transactions in Silver", non_success)
]

dq_df = (
    spark.createDataFrame(
        dq_results,
        ["check_name", "failed_records"]
    )
    .withColumn(
        "check_status",
        F.when(
            F.col("failed_records") == 0,
            "PASS"
        ).otherwise("FAIL")
    )
    .withColumn(
        "check_timestamp",
        F.current_timestamp()
    )
)

display(dq_df)

check_name,failed_records,check_status,check_timestamp
Silver duplicate txn_id,0,PASS,2026-08-22T06:54:35.225Z
Invalid amounts in Silver,0,PASS,2026-08-22T06:54:35.225Z
Invalid currencies in Silver,0,PASS,2026-08-22T06:54:35.225Z
Invalid txn types in Silver,0,PASS,2026-08-22T06:54:35.225Z
Non-success transactions in Silver,0,PASS,2026-08-22T06:54:35.225Z


##Save DQ results

In [0]:
(
    dq_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.data_quality_results")
)

print("Data quality report saved")

Data quality report saved
